# Master Thesis Experiments

Notebook for the experiment log, setup steps, and result notes for the master thesis.


## E0 - Valid max trial step count

Goal:
- estimate a reasonable maximum number of trial steps for successful rollouts
- use a deterministic evaluation setup so the step distribution is reproducible
- sample a fixed subset of anatomies first, then reuse that subset for the experiment

Configuration:
- `policy_mode = deterministic`
- `stochastic_environment_mode = random_start`
- `max_episode_steps = 1000`
- `n_anatomies = 12`
- `targets = 3 random targets per anatomy`
- `trials_per_cell = 100`


### E0.1 Sample 12 anatomies

This first step draws 12 anatomies uniformly at random from the anatomy registry and overwrites the experimental prep file for E0.


In [7]:
from pathlib import Path
import json
import sys

from IPython.display import Markdown, display

ROOT = Path('../../').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from steve_recommender.eval_v2.experimental_prep_scripts.sample_anatomies import sample_anatomies_from_registry
from steve_recommender.eval_v2.service import DefaultEvaluationService

POOL = Path('../../data/anatomy_registry').resolve()
OUT = Path('../../results/experimental_prep/sample_12.json').resolve()
POOL_INDEX = POOL / 'index.json'

payload = sample_anatomies_from_registry(
    pool_path=POOL,
    n=12,
    seed=123,
    strata='none',
    sampling_method='random',
    branches=('bct', 'lcca', 'lsa'),
    workers=4,
)

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')

service = DefaultEvaluationService()
rows = []
for item in payload['selected_anatomies']:
    anatomy = service.get_anatomy(record_id=item['record_id'], registry_path=POOL_INDEX)
    branches = service.list_branches(anatomy)
    rows.append(
        {
            'record_id': item['record_id'],
            'arch_type': item['arch_type'],
            'seed': item['seed'],
            'target_branches': ', '.join(branch.name for branch in branches),
            'tortuosity': item['tortuosity'],
            'stratum': item['stratum'],
        }
    )

table_lines = [
    '| record_id | arch_type | seed | target_branches | tortuosity | stratum |',
    '|---|---|---:|---|---:|---|',
]
for row in rows:
    table_lines.append(
        '| {record_id} | {arch_type} | {seed} | {target_branches} | {tortuosity:.6f} | {stratum} |'.format(**row)
    )

display(Markdown('\n'.join(table_lines)))
print(json.dumps(rows, indent=2, sort_keys=True))


| record_id | arch_type | seed | target_branches | tortuosity | stratum |
|---|---|---:|---|---:|---|
| Tree_625 | I | 1248887895 | aorta, bct, lcca, lsa, rcca, rsa | 0.018721 | all |
| Tree_857 | I | 659745269 | aorta, bct, lcca, lsa, rcca, rsa | 0.027128 | all |
| Tree_1428 | I | 2105833333 | aorta, bct, lcca, lsa, rcca, rsa | 0.015694 | all |
| Tree_1764 | I | 1577845490 | aorta, bct, lcca, lsa, rcca, rsa | 0.016744 | all |
| Tree_4367 | I | 57901021 | aorta, bct, lcca, lsa, rcca, rsa | 0.027405 | all |
| Tree_4385 | I | 1671607265 | aorta, bct, lcca, lsa, rcca, rsa | 0.022011 | all |
| Tree_5442 | I | 503907874 | aorta, bct, lcca, lsa, rcca, rsa | 0.024660 | all |
| Tree_5583 | I | 797089154 | aorta, bct, lcca, lsa, rcca, rsa | 0.024165 | all |
| Tree_6211 | I | 1096800833 | aorta, bct, lcca, lsa, rcca, rsa | 0.028689 | all |
| Tree_6672 | I | 1288778203 | aorta, bct, lcca, lsa, rcca, rsa | 0.027289 | all |
| Tree_8785 | I | 1291750865 | aorta, bct, lcca, lsa, rcca, rsa | 0.024043 | all |
| Tree_9213 | I | 908489492 | aorta, bct, lcca, lsa, rcca, rsa | 0.029544 | all |

[
  {
    "arch_type": "I",
    "record_id": "Tree_625",
    "seed": 1248887895,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.018721201833
  },
  {
    "arch_type": "I",
    "record_id": "Tree_857",
    "seed": 659745269,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.027127724525
  },
  {
    "arch_type": "I",
    "record_id": "Tree_1428",
    "seed": 2105833333,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.015694332837
  },
  {
    "arch_type": "I",
    "record_id": "Tree_1764",
    "seed": 1577845490,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.016743728769
  },
  {
    "arch_type": "I",
    "record_id": "Tree_4367",
    "seed": 57901021,
    "stratum": "all",
    "target_branches": "aorta, bct, lcca, lsa, rcca, rsa",
    "tortuosity": 0.027405191123
  },
  {
    

### E0.2 Experiment runner script

The notebook prepares the anatomy subset only. The actual eval_v2 runs are started from the standalone script:

`experiments/master-thesis/run_e0_experiment.sh`

Example:

```bash
bash experiments/master-thesis/run_e0_experiment.sh
```


### E0.3 Show experiment runner source

This cell prints the current `run_e0_experiment.sh` source so the notebook contains the full experiment setup in one place.


In [3]:
from pathlib import Path

script_path = Path('run_e0_experiment.sh')
print(script_path.read_text(encoding='utf-8'))


from __future__ import annotations

import argparse
import json
import random
import subprocess
import sys
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

PROJECT_ROOT = Path(__file__).resolve().parents[2]
DEFAULT_SAMPLE_JSON = PROJECT_ROOT / "results" / "experimental_prep" / "sample_12.json"
DEFAULT_OUTPUT_ROOT = PROJECT_ROOT / "results" / "master_thesis" / "e0"
DEFAULT_TARGET_BRANCHES = ("bct", "lcca", "lsa")


def _parse_csv_list(text: str) -> Tuple[str, ...]:
    values = tuple(item.strip() for item in str(text).split(",") if item.strip())
    if not values:
        raise ValueError("List argument must not be empty")
    return values


def _parse_wire_ref(text: str) -> Tuple[str, str]:
    parts = tuple(part.strip() for part in str(text).split("/", maxsplit=1))
    if len(parts) != 2 or not parts[0] or not parts[1]:
        raise ValueError(f"wire ref must look like 'model/wire', got {text!r}")
    return parts[0], parts[1]




E0.5 "how should we constitute an evaluation score?"
Config 1: Deterministic, N=1. One trial per evaluation. Policy is deterministic, environment is whatever it is at start. Score is just that one trial's score. No aggregation.
Config 2: Deterministic, N=k, varied env_seed. k trials, all with deterministic policy but different env_seed per trial. Aggregate (IQM) over the k trials. The variance in the aggregate comes from environment perturbations alone.
Config 3: Stochastic, N=k, fixed env_seed. k trials, all with the same physical starting state but different policy_seed. Aggregate over the k trials. Variance comes from action sampling alone.
Config 4: Stochastic, N=k, varied env_seed. k trials, both policy and environment seeds vary. Variance from both sources combined.



G1. Establish whether stEVE-simulated RL expert agents constitute a stable measurement system whose outputs (a) support a methodology for clinically-grounded guidewire ranking, and (b) function as a benchmark environment for evaluating autonomous endovascular navigation methods on clinical outcomes rather than success rate alone, by characterising the system's stability under stochastic perturbation, its discriminability across wires, and its sensitivity to the reward function used during agent training.


H1 — Stability (within-anatomy ranking reliability).
The wire ranking produced by the system on a given anatomy is stable across stochastic perturbation. Operationally: Kendall's τ between rankings produced from independent seed splits exceeds a pre-registered threshold (suggested: τ ≥ 0.7) for the majority of anatomies tested.
H1 falsified ⟹ Contribution 1 substantially weakened (you can't recommend a wire if the ranking flips between runs); Contribution 2 partially weakened (a noisy benchmark is still useful but less so).


H2 — Discriminability (the wires are actually distinguishable).
The 15 wires resolve into multiple statistically distinguishable tiers under the eval scoring. Operationally: pairwise probability-of-improvement matrix shows ≥ k tiers (pre-register k = 3) with within-tier P(A > B) near 0.5 and between-tier P > 0.75.
H2 falsified ⟹ either the wire matrix is too narrow to differentiate (Contribution 1 has limited scope but isn't wrong), or the eval metric is too coarse (Contribution 2 needs work). Diagnostic in either direction.


H3 — Reward sensitivity (the expert-agent assumption holds, or doesn't).
Rankings produced by task-reward agents (Set A) and force-aware agents (Set B) on the same anatomies and targets agree above a pre-registered threshold (suggested: τ ≥ 0.6 between ranking pairs).
H3 confirmed ⟹ rankings are robust to reward choice; the expert-agent assumption is approximately correct. Contribution 1 strengthened. Contribution 2 still useful but less differentiating.
H3 falsified ⟹ rankings depend on training reward in a structured way. This is a finding, not a failure: Contribution 1 must be reframed as "ranking under specified reward" rather than "the ranking of the wire," and Contribution 2 is strengthened because the benchmark can detect reward-design differences.
Either outcome is publishable. This is what we want from a hypothesis.


H4 — Cross-anatomy structure (does anatomy matter, and how).
Cross-anatomy ranking similarity (the τ-matrix between anatomies' rankings) correlates with anatomical features in a mechanically interpretable way. Specifically: anatomies similar in tortuosity / branch geometry produce similar rankings; anatomies dissimilar in those features produce dissimilar rankings.


H4 confirmed ⟹ the system captures real mechanical structure; anatomy-conditional recommendation is in principle feasible (future work).
H4 partially confirmed (some structure but noisy) ⟹ the methodology has resolution limits at fine anatomical distinctions; useful contribution but bounded.
H4 falsified (rankings vary randomly across anatomies) ⟹ either anatomy descriptors are wrong, or the system isn't capturing mechanics, or both. Diagnostic.
H4 is where your old H1/H2/H3 mechanistic content lives — but now as explanatory analysis of the cross-anatomy structure, not as primary claims.
